In [1]:
import os

import polars as pl

import plotly.express as px

from pytecgg import GNSSContext
from pytecgg.parsing import read_rinex_nav, read_rinex_obs
from pytecgg.utils import summarise_rinex_data, download_nav_bkg
from pytecgg.satellites import prepare_ephemeris, satellite_coordinates, calculate_ipp
from pytecgg.linear_combinations import calculate_linear_combinations
from pytecgg.tec_calibration import extract_arcs, calculate_tec, calculate_vertical_equivalent

OBS_NAV_DIR = 'data/calm-obs-nav'

In [2]:
def get_station_id(fid: str) -> str:
    for i, c in enumerate(fid):
        if c.isnumeric():
            return fid[:i]
    raise Exception("No station ID found")

file_ids = set([f[:-1] for f in os.listdir(OBS_NAV_DIR) if f.endswith('.25o') or f.endswith('.25n')])
obs_files = [f"{f}o" for f in file_ids]
nav_files = [f"{f}n" for f in file_ids]

In [7]:
f_idx = 0
nav_dict = read_rinex_nav(os.path.join(OBS_NAV_DIR, nav_files[f_idx]))
df_obs, rec_pos, rinex_version = read_rinex_obs(os.path.join(OBS_NAV_DIR, obs_files[f_idx]))
ctx = GNSSContext(
    receiver_pos=rec_pos,
    receiver_name=get_station_id(list(file_ids)[f_idx]),
    rinex_version=rinex_version,
    h_ipp=350_000,
    # systems=['G', 'E', 'R', 'C'],
    # systems=['G', 'E', 'R'],
    systems=['G', 'E'],
)
summarise_rinex_data(df_obs, nav_dict)

🛰️  GNSS RINEX DATA SUMMARY

📅 Temporal Coverage (OBS)

   - Start:    2025-11-28 00:00:00+00:00
   - End:      2025-11-28 23:59:45+00:00
   - Duration: 23:59:45
   - Sampling: 15.0s

📡 Constellations Breakdown (OBS)

Sys  | SVs  | Total Records   | Available Signals
-------------------------------------------------
E    | 26   | 409,089         | C1, C5, C7, C8, L1, L5, L7, L8, S1, S5, S7, S8
G    | 32   | 414,026         | C1, C2, C5, L1, L2, L5, P2, S1, S2, S5
R    | 24   | 248,234         | C1, C2, C7, L1, L2, L7, P1, S1, S2, S7
S    | 1    | 23,748          | C1, C5, L1, L5, S1, S5

📖 Ephemeris Coverage (NAV)

   - GPS     :  32 satellites have at least an ephemeris


In [8]:
df_obs

epoch,sv,observable,value
"datetime[μs, UTC]",str,str,f64
2025-11-28 00:00:00 UTC,"""G19""","""L1""",1.0718e8
2025-11-28 00:00:00 UTC,"""G19""","""L2""",8.3520e7
2025-11-28 00:00:00 UTC,"""G19""","""C1""",2.0396e7
2025-11-28 00:00:00 UTC,"""G19""","""P2""",2.0396e7
2025-11-28 00:00:00 UTC,"""G19""","""S1""",49.5
…,…,…,…
2025-11-28 23:59:45 UTC,"""R09""","""C2""",2.3539e7
2025-11-28 23:59:45 UTC,"""R19""","""L1""",1.1496e8
2025-11-28 23:59:45 UTC,"""R19""","""L2""",8.9415e7


In [9]:
ephem_dict = prepare_ephemeris(nav_dict, ctx=ctx)
ephem_dict

{'G02': {'constellation': 'GPS',
  'sv': 'G02',
  'datetime': datetime.datetime(2025, 11, 28, 18, 0, tzinfo=zoneinfo.ZoneInfo(key='UTC')),
  'gps_week': 2394,
  'gps_seconds': 496800.0,
  'accuracy': 2.0,
  'cic': -3.557652235031e-07,
  'cis': -1.359730958939e-07,
  'clock_bias': -1.735985279083e-05,
  'clock_drift': 8.526512829121e-12,
  'clock_drift_rate': 0.0,
  'crc': 212.8125,
  'crs': -7.0625,
  'cuc': -3.501772880554e-07,
  'cus': 9.017065167427e-06,
  'deltaN': 4.294464595974e-09,
  'e': 0.0168668590486,
  'fitInt': 4.0,
  'health': 0.0,
  'i0': 0.9639867186705,
  'idot': -7.214586230761e-11,
  'iodc': 93.0,
  'iode': 93.0,
  'l2Codes': 1.0,
  'l2p': 0.0,
  'm0': -0.6709782757956,
  'omega': -0.889837087595,
  'omega0': -1.556433413185,
  'omegaDot': -8.166411592393e-09,
  'sqrta': 5153.629720688,
  't_tm': 489618.0,
  'tgd': -1.769512891769e-08,
  'toe': 496800.0},
 'G20': {'constellation': 'GPS',
  'sv': 'G20',
  'datetime': datetime.datetime(2025, 11, 28, 8, 0, tzinfo=zonein

In [10]:
# Compute selected linear combinations
df_lc = calculate_linear_combinations(
    df_obs,
    ctx=ctx,
    combinations=["gflc_phase", "gflc_code", "mw"],
    selection_mode="quality",
    # band_overrides={"G": ("L1", "L2")},
    # band_overrides={"G": ("L1", "L2"), "S": ("L1", "L5")}
)

df_lc

epoch,sv,gflc_phase,gflc_code,mw
"datetime[μs, UTC]",str,f64,f64,f64
2025-11-28 00:00:00 UTC,"""G19""",-238.459119,63.654738,-23.56864
2025-11-28 00:00:00 UTC,"""G01""",-120.327384,103.286665,-14.951112
2025-11-28 00:00:00 UTC,"""G13""",-153.094106,89.485922,-49.284904
2025-11-28 00:00:00 UTC,"""G17""",-236.224711,75.066525,6.856725
2025-11-28 00:00:00 UTC,"""G22""",-173.034919,76.475153,-24.895292
…,…,…,…,…
2025-11-28 23:59:30 UTC,"""E02""",-199.732747,62.640294,-56.778496
2025-11-28 23:59:30 UTC,"""E27""",-207.027554,56.547031,-58.996317
2025-11-28 23:59:30 UTC,"""E07""",-156.208125,66.645547,-42.201514


In [13]:
# Identify continuous arcs, detect cycle slips
# and loss-of-lock events
df_arcs = extract_arcs(
    df=df_lc,
    ctx=ctx,
    threshold_abs=10,
    threshold_std=10,
    threshold_jump=5
)

df_arcs

epoch,sv,gflc_phase,gflc_code,mw,is_loss_of_lock,is_cycle_slip,id_arc,id_arc_valid,gflc_phase_fix,gflc_code_fix,mw_fix,gflc_levelled
"datetime[μs, UTC]",str,f64,f64,f64,bool,bool,str,str,f64,f64,f64,f64
2025-11-28 00:00:00 UTC,"""G19""",-238.459119,63.654738,-23.56864,false,false,"""bdla_g19_20251128_001""","""bdla_g19_20251128_001""",-238.459119,63.654738,-23.56864,55.671883
2025-11-28 00:00:00 UTC,"""G01""",-120.327384,103.286665,-14.951112,false,false,"""bdla_g01_20251128_001""","""bdla_g01_20251128_001""",-120.327384,103.286665,-14.951112,103.441579
2025-11-28 00:00:00 UTC,"""G13""",-153.094106,89.485922,-49.284904,false,false,"""bdla_g13_20251128_001""","""bdla_g13_20251128_001""",-153.094106,89.485922,-49.284904,71.7088
2025-11-28 00:00:00 UTC,"""G17""",-236.224711,75.066525,6.856725,false,false,"""bdla_g17_20251128_001""","""bdla_g17_20251128_001""",-236.224711,75.066525,6.856725,90.875799
2025-11-28 00:00:00 UTC,"""G22""",-173.034919,76.475153,-24.895292,false,false,"""bdla_g22_20251128_001""","""bdla_g22_20251128_001""",-173.034919,76.475153,-24.895292,104.269185
…,…,…,…,…,…,…,…,…,…,…,…,…
2025-11-28 23:59:30 UTC,"""E02""",-199.732747,62.640294,-56.778496,false,false,"""bdla_e02_20251128_003""","""bdla_e02_20251128_003""",-199.732747,53.72162,-56.778496,60.979762
2025-11-28 23:59:30 UTC,"""E27""",-207.027554,56.547031,-58.996317,false,false,"""bdla_e27_20251128_011""","""bdla_e27_20251128_011""",-207.027554,56.112353,-58.996317,52.090205
2025-11-28 23:59:30 UTC,"""E07""",-156.208125,66.645547,-42.201514,false,false,"""bdla_e07_20251128_003""","""bdla_e07_20251128_003""",-156.208125,60.948152,-42.201514,64.003821


In [14]:
df_arcs.filter(
    pl.col('is_loss_of_lock')
)

epoch,sv,gflc_phase,gflc_code,mw,is_loss_of_lock,is_cycle_slip,id_arc,id_arc_valid,gflc_phase_fix,gflc_code_fix,mw_fix,gflc_levelled
"datetime[μs, UTC]",str,f64,f64,f64,bool,bool,str,str,f64,f64,f64,f64
2025-11-28 00:15:15 UTC,"""G11""",null,null,null,true,null,"""bdla_g11_20251128_002""",null,null,null,null,null
2025-11-28 00:15:30 UTC,"""G11""",null,null,null,true,null,"""bdla_g11_20251128_003""",null,null,null,null,null
2025-11-28 00:15:45 UTC,"""G11""",null,null,null,true,null,"""bdla_g11_20251128_004""",null,null,null,null,null
2025-11-28 00:16:00 UTC,"""G11""",null,null,null,true,null,"""bdla_g11_20251128_005""",null,null,null,null,null
2025-11-28 00:16:15 UTC,"""G11""",null,null,null,true,null,"""bdla_g11_20251128_006""",null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…
2025-11-28 21:48:15 UTC,"""E07""",-148.171527,77.473702,-42.236977,true,null,"""bdla_e07_20251128_003""","""bdla_e07_20251128_003""",-148.171527,77.473702,-42.236977,72.040418
2025-11-28 21:48:15 UTC,"""E08""",-87.590191,54.909224,-27.870521,true,null,"""bdla_e08_20251128_003""","""bdla_e08_20251128_003""",-87.590191,54.909224,-27.870521,52.938832
2025-11-28 21:48:15 UTC,"""E30""",-207.050609,62.911968,-58.484262,true,null,"""bdla_e30_20251128_003""","""bdla_e30_20251128_003""",-207.050609,62.911968,-58.484262,64.71213


In [15]:
df_arcs.filter(
    pl.col('is_cycle_slip')
)

epoch,sv,gflc_phase,gflc_code,mw,is_loss_of_lock,is_cycle_slip,id_arc,id_arc_valid,gflc_phase_fix,gflc_code_fix,mw_fix,gflc_levelled
"datetime[μs, UTC]",str,f64,f64,f64,bool,bool,str,str,f64,f64,f64,f64
2025-11-28 00:34:45 UTC,"""G07""",-89.974497,144.365291,-50.844586,false,true,"""bdla_g07_20251128_001""","""bdla_g07_20251128_001""",-6.783127,147.924931,-19.291133,145.975293
2025-11-28 00:37:30 UTC,"""G07""",-29.991348,138.711745,-31.521763,false,true,"""bdla_g07_20251128_001""","""bdla_g07_20251128_001""",-1.011809,149.54295,-18.973719,151.746611
2025-11-28 01:39:30 UTC,"""G30""",-151.507764,167.008028,-65.249645,false,true,"""bdla_g30_20251128_001""","""bdla_g30_20251128_001""",-54.498357,122.226996,-28.724777,145.777292
2025-11-28 02:34:00 UTC,"""G25""",-107.843302,119.866593,-50.621986,false,true,"""bdla_g25_20251128_001""",null,null,null,null,null
2025-11-28 03:15:15 UTC,"""G17""",-1238.07052,63.502454,-463.122212,false,true,"""bdla_g17_20251128_001""","""bdla_g17_20251128_001""",-251.03068,101.897073,7.013141,76.06983
…,…,…,…,…,…,…,…,…,…,…,…,…
2025-11-28 01:39:30 UTC,"""E19""",-223.475132,157.182894,-102.567426,false,true,"""bdla_e19_20251128_002""","""bdla_e19_20251128_002""",169.016313,130.651974,46.421855,136.839098
2025-11-28 02:34:30 UTC,"""E13""",265.106122,59.550971,138.994227,false,true,"""bdla_e13_20251128_001""","""bdla_e13_20251128_001""",-282.669526,42.536408,-69.111203,40.880086
2025-11-28 17:55:00 UTC,"""E34""",-502.427048,180.896165,-242.929395,false,true,"""bdla_e34_20251128_002""","""bdla_e34_20251128_002""",100.216057,121.896305,-13.814765,177.543367


In [16]:
df_arcs.filter(
    pl.col('id_arc_valid').is_null()
)

epoch,sv,gflc_phase,gflc_code,mw,is_loss_of_lock,is_cycle_slip,id_arc,id_arc_valid,gflc_phase_fix,gflc_code_fix,mw_fix,gflc_levelled
"datetime[μs, UTC]",str,f64,f64,f64,bool,bool,str,str,f64,f64,f64,f64
2025-11-28 00:15:15 UTC,"""G11""",null,null,null,true,null,"""bdla_g11_20251128_002""",null,null,null,null,null
2025-11-28 00:15:30 UTC,"""G11""",null,null,null,true,null,"""bdla_g11_20251128_003""",null,null,null,null,null
2025-11-28 00:15:45 UTC,"""G11""",null,null,null,true,null,"""bdla_g11_20251128_004""",null,null,null,null,null
2025-11-28 00:16:00 UTC,"""G11""",null,null,null,true,null,"""bdla_g11_20251128_005""",null,null,null,null,null
2025-11-28 00:16:15 UTC,"""G11""",null,null,null,true,null,"""bdla_g11_20251128_006""",null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…
2025-11-28 15:30:00 UTC,"""E06""",null,null,null,true,null,"""bdla_e06_20251128_010""",null,null,null,null,null
2025-11-28 15:30:15 UTC,"""E06""",null,null,null,true,null,"""bdla_e06_20251128_011""",null,null,null,null,null
2025-11-28 15:30:30 UTC,"""E06""",null,null,null,true,null,"""bdla_e06_20251128_012""",null,null,null,null,null


In [17]:
# Compute satellites' ECEF coordinates
df_coords = satellite_coordinates(
    sv_ids=df_arcs["sv"],
    epochs=df_arcs["epoch"],
    ephem_dict=ephem_dict
)

df_coords

ℹ️  These may be missing from NAV file or have been excluded due to invalid/incomplete parameters.



sv,epoch,sat_x,sat_y,sat_z
str,"datetime[μs, UTC]",f64,f64,f64
"""G19""",2025-11-28 00:00:00 UTC,-1.4480e7,-1.9940e7,1.0463e7
"""G01""",2025-11-28 00:00:00 UTC,1.4128e7,-1.3009e7,1.8327e7
"""G13""",2025-11-28 00:00:00 UTC,-2.2396e7,-1.3092e7,6.0228e6
"""G17""",2025-11-28 00:00:00 UTC,-7.8323e6,-1.9376e7,1.6631e7
"""G22""",2025-11-28 00:00:00 UTC,-4.1329e6,-1.4993e7,2.1891e7
…,…,…,…,…
"""G07""",2025-11-28 23:59:45 UTC,5.8548e6,-2.4128e7,-8.2386e6
"""G06""",2025-11-28 23:59:45 UTC,-5.2602e6,-2.4393e7,-8.8174e6
"""G01""",2025-11-28 23:59:45 UTC,1.4119e7,-1.2462e7,1.8711e7


In [18]:
# Join coordinates with the observation data
df_geom = df_arcs.join(df_coords, on=["sv", "epoch"], how="left")
df_geom

epoch,sv,gflc_phase,gflc_code,mw,is_loss_of_lock,is_cycle_slip,id_arc,id_arc_valid,gflc_phase_fix,gflc_code_fix,mw_fix,gflc_levelled,sat_x,sat_y,sat_z
"datetime[μs, UTC]",str,f64,f64,f64,bool,bool,str,str,f64,f64,f64,f64,f64,f64,f64
2025-11-28 00:00:00 UTC,"""G19""",-238.459119,63.654738,-23.56864,false,false,"""bdla_g19_20251128_001""","""bdla_g19_20251128_001""",-238.459119,63.654738,-23.56864,55.671883,-1.4480e7,-1.9940e7,1.0463e7
2025-11-28 00:00:00 UTC,"""G01""",-120.327384,103.286665,-14.951112,false,false,"""bdla_g01_20251128_001""","""bdla_g01_20251128_001""",-120.327384,103.286665,-14.951112,103.441579,1.4128e7,-1.3009e7,1.8327e7
2025-11-28 00:00:00 UTC,"""G13""",-153.094106,89.485922,-49.284904,false,false,"""bdla_g13_20251128_001""","""bdla_g13_20251128_001""",-153.094106,89.485922,-49.284904,71.7088,-2.2396e7,-1.3092e7,6.0228e6
2025-11-28 00:00:00 UTC,"""G17""",-236.224711,75.066525,6.856725,false,false,"""bdla_g17_20251128_001""","""bdla_g17_20251128_001""",-236.224711,75.066525,6.856725,90.875799,-7.8323e6,-1.9376e7,1.6631e7
2025-11-28 00:00:00 UTC,"""G22""",-173.034919,76.475153,-24.895292,false,false,"""bdla_g22_20251128_001""","""bdla_g22_20251128_001""",-173.034919,76.475153,-24.895292,104.269185,-4.1329e6,-1.4993e7,2.1891e7
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2025-11-28 23:59:30 UTC,"""E02""",-199.732747,62.640294,-56.778496,false,false,"""bdla_e02_20251128_003""","""bdla_e02_20251128_003""",-199.732747,53.72162,-56.778496,60.979762,NaN,NaN,NaN
2025-11-28 23:59:30 UTC,"""E27""",-207.027554,56.547031,-58.996317,false,false,"""bdla_e27_20251128_011""","""bdla_e27_20251128_011""",-207.027554,56.112353,-58.996317,52.090205,NaN,NaN,NaN
2025-11-28 23:59:30 UTC,"""E07""",-156.208125,66.645547,-42.201514,false,false,"""bdla_e07_20251128_003""","""bdla_e07_20251128_003""",-156.208125,60.948152,-42.201514,64.003821,NaN,NaN,NaN


In [19]:
df_geom

epoch,sv,gflc_phase,gflc_code,mw,is_loss_of_lock,is_cycle_slip,id_arc,id_arc_valid,gflc_phase_fix,gflc_code_fix,mw_fix,gflc_levelled,sat_x,sat_y,sat_z
"datetime[μs, UTC]",str,f64,f64,f64,bool,bool,str,str,f64,f64,f64,f64,f64,f64,f64
2025-11-28 00:00:00 UTC,"""G19""",-238.459119,63.654738,-23.56864,false,false,"""bdla_g19_20251128_001""","""bdla_g19_20251128_001""",-238.459119,63.654738,-23.56864,55.671883,-1.4480e7,-1.9940e7,1.0463e7
2025-11-28 00:00:00 UTC,"""G01""",-120.327384,103.286665,-14.951112,false,false,"""bdla_g01_20251128_001""","""bdla_g01_20251128_001""",-120.327384,103.286665,-14.951112,103.441579,1.4128e7,-1.3009e7,1.8327e7
2025-11-28 00:00:00 UTC,"""G13""",-153.094106,89.485922,-49.284904,false,false,"""bdla_g13_20251128_001""","""bdla_g13_20251128_001""",-153.094106,89.485922,-49.284904,71.7088,-2.2396e7,-1.3092e7,6.0228e6
2025-11-28 00:00:00 UTC,"""G17""",-236.224711,75.066525,6.856725,false,false,"""bdla_g17_20251128_001""","""bdla_g17_20251128_001""",-236.224711,75.066525,6.856725,90.875799,-7.8323e6,-1.9376e7,1.6631e7
2025-11-28 00:00:00 UTC,"""G22""",-173.034919,76.475153,-24.895292,false,false,"""bdla_g22_20251128_001""","""bdla_g22_20251128_001""",-173.034919,76.475153,-24.895292,104.269185,-4.1329e6,-1.4993e7,2.1891e7
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2025-11-28 23:59:30 UTC,"""E02""",-199.732747,62.640294,-56.778496,false,false,"""bdla_e02_20251128_003""","""bdla_e02_20251128_003""",-199.732747,53.72162,-56.778496,60.979762,NaN,NaN,NaN
2025-11-28 23:59:30 UTC,"""E27""",-207.027554,56.547031,-58.996317,false,false,"""bdla_e27_20251128_011""","""bdla_e27_20251128_011""",-207.027554,56.112353,-58.996317,52.090205,NaN,NaN,NaN
2025-11-28 23:59:30 UTC,"""E07""",-156.208125,66.645547,-42.201514,false,false,"""bdla_e07_20251128_003""","""bdla_e07_20251128_003""",-156.208125,60.948152,-42.201514,64.003821,NaN,NaN,NaN


In [20]:
df_final = calculate_ipp(df_geom, ctx=ctx, min_elevation=20)
df_final

epoch,sv,gflc_phase,gflc_code,mw,is_loss_of_lock,is_cycle_slip,id_arc,id_arc_valid,gflc_phase_fix,gflc_code_fix,mw_fix,gflc_levelled,sat_x,sat_y,sat_z,lat_ipp,lon_ipp,azi,ele
"datetime[μs, UTC]",str,f64,f64,f64,bool,bool,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2025-11-28 00:00:00 UTC,"""G19""",-238.459119,63.654738,-23.56864,false,false,"""bdla_g19_20251128_001""","""bdla_g19_20251128_001""",-238.459119,63.654738,-23.56864,55.671883,-1.4480e7,-1.9940e7,1.0463e7,28.595508,-114.451428,244.914092,73.470416
2025-11-28 00:00:00 UTC,"""G13""",-153.094106,89.485922,-49.284904,false,false,"""bdla_g13_20251128_001""","""bdla_g13_20251128_001""",-153.094106,89.485922,-49.284904,71.7088,-2.2396e7,-1.3092e7,6.0228e6,27.985744,-116.898429,252.377501,42.810822
2025-11-28 00:00:00 UTC,"""G17""",-236.224711,75.066525,6.856725,false,false,"""bdla_g17_20251128_001""","""bdla_g17_20251128_001""",-236.224711,75.066525,6.856725,90.875799,-7.8323e6,-1.9376e7,1.6631e7,29.634616,-113.451592,7.184559,77.340856
2025-11-28 00:00:00 UTC,"""G22""",-173.034919,76.475153,-24.895292,false,false,"""bdla_g22_20251128_001""","""bdla_g22_20251128_001""",-173.034919,76.475153,-24.895292,104.269185,-4.1329e6,-1.4993e7,2.1891e7,30.921207,-113.122213,10.643537,56.047317
2025-11-28 00:00:00 UTC,"""G14""",-236.216108,79.492281,-75.272058,false,false,"""bdla_g14_20251128_001""","""bdla_g14_20251128_001""",-236.216108,79.492281,-75.272058,60.016566,1.7107e6,-1.7381e7,2.0009e7,30.745411,-111.812003,40.024408,51.535215
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2025-11-28 23:59:30 UTC,"""E02""",-199.732747,62.640294,-56.778496,false,false,"""bdla_e02_20251128_003""","""bdla_e02_20251128_003""",-199.732747,53.72162,-56.778496,60.979762,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-11-28 23:59:30 UTC,"""E27""",-207.027554,56.547031,-58.996317,false,false,"""bdla_e27_20251128_011""","""bdla_e27_20251128_011""",-207.027554,56.112353,-58.996317,52.090205,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-11-28 23:59:30 UTC,"""E07""",-156.208125,66.645547,-42.201514,false,false,"""bdla_e07_20251128_003""","""bdla_e07_20251128_003""",-156.208125,60.948152,-42.201514,64.003821,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [21]:
df_final.filter(
    pl.col("id_arc_valid").is_not_null()
).select(
    [
        "epoch",
        "sv",
        "id_arc_valid",
        "gflc_levelled",
        "lat_ipp",
        "lon_ipp",
        "azi",
        "ele",
    ]
)

epoch,sv,id_arc_valid,gflc_levelled,lat_ipp,lon_ipp,azi,ele
"datetime[μs, UTC]",str,str,f64,f64,f64,f64,f64
2025-11-28 00:00:00 UTC,"""G19""","""bdla_g19_20251128_001""",55.671883,28.595508,-114.451428,244.914092,73.470416
2025-11-28 00:00:00 UTC,"""G13""","""bdla_g13_20251128_001""",71.7088,27.985744,-116.898429,252.377501,42.810822
2025-11-28 00:00:00 UTC,"""G17""","""bdla_g17_20251128_001""",90.875799,29.634616,-113.451592,7.184559,77.340856
2025-11-28 00:00:00 UTC,"""G22""","""bdla_g22_20251128_001""",104.269185,30.921207,-113.122213,10.643537,56.047317
2025-11-28 00:00:00 UTC,"""G14""","""bdla_g14_20251128_001""",60.016566,30.745411,-111.812003,40.024408,51.535215
…,…,…,…,…,…,…,…
2025-11-28 23:59:30 UTC,"""E02""","""bdla_e02_20251128_003""",60.979762,NaN,NaN,NaN,NaN
2025-11-28 23:59:30 UTC,"""E27""","""bdla_e27_20251128_011""",52.090205,NaN,NaN,NaN,NaN
2025-11-28 23:59:30 UTC,"""E07""","""bdla_e07_20251128_003""",64.003821,NaN,NaN,NaN,NaN


In [31]:
from pytecgg.tec_calibration import calculate_tec

df_calibrated = calculate_tec(
    df_final,
    ctx=ctx,
    max_polynomial_degree=3,
    batch_size_epochs=30
)
df_calibrated = df_calibrated.sort(by='epoch')
df_calibrated

epoch,sv,gflc_phase,gflc_code,mw,is_loss_of_lock,is_cycle_slip,id_arc,id_arc_valid,gflc_phase_fix,gflc_code_fix,mw_fix,gflc_levelled,sat_x,sat_y,sat_z,lat_ipp,lon_ipp,azi,ele,bias,stec,vtec
"datetime[μs, UTC]",str,f64,f64,f64,bool,bool,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2025-11-28 00:00:00 UTC,"""G19""",-238.459119,63.654738,-23.56864,false,false,"""bdla_g19_20251128_001""","""bdla_g19_20251128_001""",-238.459119,63.654738,-23.56864,55.671883,-1.4480e7,-1.9940e7,1.0463e7,28.595508,-114.451428,244.914092,73.470416,29.112002,26.559882,25.575731
2025-11-28 00:00:00 UTC,"""G13""",-153.094106,89.485922,-49.284904,false,false,"""bdla_g13_20251128_001""","""bdla_g13_20251128_001""",-153.094106,89.485922,-49.284904,71.7088,-2.2396e7,-1.3092e7,6.0228e6,27.985744,-116.898429,252.377501,42.810822,27.44149,44.26731,31.811553
2025-11-28 00:00:00 UTC,"""G17""",-236.224711,75.066525,6.856725,false,false,"""bdla_g17_20251128_001""","""bdla_g17_20251128_001""",-236.224711,75.066525,6.856725,90.875799,-7.8323e6,-1.9376e7,1.6631e7,29.634616,-113.451592,7.184559,77.340856,69.025228,21.850571,21.373889
2025-11-28 00:00:00 UTC,"""G22""",-173.034919,76.475153,-24.895292,false,false,"""bdla_g22_20251128_001""","""bdla_g22_20251128_001""",-173.034919,76.475153,-24.895292,104.269185,-4.1329e6,-1.4993e7,2.1891e7,30.921207,-113.122213,10.643537,56.047317,81.966633,22.302552,18.920543
2025-11-28 00:00:00 UTC,"""G14""",-236.216108,79.492281,-75.272058,false,false,"""bdla_g14_20251128_001""","""bdla_g14_20251128_001""",-236.216108,79.492281,-75.272058,60.016566,1.7107e6,-1.7381e7,2.0009e7,30.745411,-111.812003,40.024408,51.535215,39.756707,20.259859,16.363192
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2025-11-28 23:59:45 UTC,"""G22""",-382.801056,63.873647,-108.550537,false,false,"""bdla_g22_20251128_032""","""bdla_g22_20251128_032""",-382.801056,217.04286,-108.550537,155.777623,-3.5590e6,-1.5249e7,2.1821e7,30.912943,-113.000117,13.653551,55.84887,125.747643,30.02998,25.425005
2025-11-28 23:59:45 UTC,"""G14""",-190.924191,76.475153,-38.820822,false,false,"""bdla_g14_20251128_003""","""bdla_g14_20251128_003""",-190.924191,175.250403,-38.820822,178.99312,2.2432e6,-1.7632e7,1.9727e7,30.710305,-111.680453,42.645757,50.916632,148.684757,30.308363,24.300637
2025-11-28 23:59:45 UTC,"""G06""",-717.116924,127.747293,-242.469967,false,false,"""bdla_g06_20251128_016""","""bdla_g06_20251128_016""",-717.116924,183.654579,-242.469967,162.66265,-5.2602e6,-2.4393e7,-8.8174e6,23.989052,-112.183919,165.862927,28.090212,102.779151,59.883499,32.835571


In [34]:
df_calibrated[['epoch', 'stec']]

epoch,stec
"datetime[μs, UTC]",f64
2025-11-28 00:00:00 UTC,26.559882
2025-11-28 00:00:00 UTC,44.26731
2025-11-28 00:00:00 UTC,21.850571
2025-11-28 00:00:00 UTC,22.302552
2025-11-28 00:00:00 UTC,20.259859
…,…
2025-11-28 23:59:45 UTC,30.02998
2025-11-28 23:59:45 UTC,30.308363
2025-11-28 23:59:45 UTC,59.883499


In [25]:
df_calibrated['sv'].unique()

sv
str
"""G13"""
"""E26"""
"""E10"""
"""G19"""
"""E29"""
…
"""G21"""
"""G22"""
"""G01"""


In [42]:
fig = px.scatter(df_calibrated['epoch', 'stec', 'id_arc_valid'], x='epoch', y='stec', render_mode='svg', color='id_arc_valid')
fig.show()

In [43]:
fig = px.scatter(df_calibrated['epoch', 'vtec', 'id_arc_valid'], x='epoch', y='vtec', render_mode='svg', color='id_arc_valid')
fig.show()